In [2]:
import numpy as np
import torch
from train_reinforce import BinPacking_Environment, get_action_from_idx
from PackingUtils import *
from ModelEMS import *
import datetime
import random
import json
import seaborn as sns
import matplotlib.pyplot as plt
import pandas
import copy

In [ ]:
item_list = {}
container_size = np.array([(35, 23, 13), (37, 26, 13), (38, 26, 13), (40, 28, 16), (42, 30, 18), (42, 30, 40), (52, 40, 17), (54, 45, 36)])
df = pandas.read_csv("task3.csv")
# print(df)
print(df.index, df.columns)

for i in df.index:
	for t in range(df.loc[i, 'qty']):
		if item_list.get(df.loc[i, 'sta_code']) is None:
			item_list[df.loc[i, 'sta_code']] = []
		item_list[df.loc[i, 'sta_code']].append(
			{
				"size": np.array([df.loc[i, '长(CM)'], df.loc[i, '宽(CM)'], df.loc[i, '高(CM)']], dtype=np.int32),
				"sku_code": df.loc[i, 'sku_code'],
			}
		)
# print(item_list)
print(container_size)
n_items = len(item_list)
print(n_items)

RangeIndex(start=0, stop=19585, step=1) Index(['sta_code', 'sku_code', '长(CM)', '宽(CM)', '高(CM)', 'qty'], dtype='object')
[[3500 2300 1300]
 [3700 2600 1300]
 [3800 2600 1300]
 [4000 2800 1600]
 [4200 3000 1800]
 [4200 3000 4000]
 [5200 4000 1700]
 [5400 4500 3600]]
6847


In [23]:
clip_num = 100
device = torch.device("cpu")
class ContainerState:
	def __init__(self, container_size):
		self.container_size = container_size
		self.height_map = np.zeros(container_size[:2])
		self.total_volume = np.prod(container_size)
		self.packed_volume = 0
	
	def add_item(self, item, position, orientation):
		new_height_map = update_height_map(
			self.height_map,
			item,
			position,
			orientation
		)
		self.packed_volume += np.prod(item)
		self.height_map = new_height_map
	
	def get_next_state(self, item, return_type='torch'):
		placement, mask = generate_EMS_and_mask(
			height_map=self.height_map, 
			coming_item=item,
			height_limit=self.container_size[2],
			clip_num=clip_num
		)
		item_state = np.array([
			item, 
			[item[1], item[0], item[2]]
		])
		# print("ENV: ", placement.shape)
		if return_type == 'torch':
			height_map = torch.Tensor(self.height_map).reshape(1, 1, self.container_dim[0], self.container_dim[1]).to(device)
			placement = torch.Tensor(placement).reshape(1, -1, 5).to(device)
			item_state = torch.Tensor(item_state).reshape(1, 2, 3).to(device)
			mask = torch.Tensor(mask).reshape(1, 2, -1).to(device)
			return placement, item_state, height_map, mask
		elif return_type == 'numpy':
			return placement, item_state, self.height_map, mask
		else:
			raise ValueError("Invalid return type")
	
	def is_full(self):
		return self.packed_volume >= self.total_volume
	
	def is_valid(self, item, action):
		li, wi, hi = item.tolist()
		x, y, r = action
		if r == 1:
			li, wi = wi, li
		# print(li, wi, hi, action)
		if x-li+1 < 0 or y+wi > self.container_size[1]:
			return False
		stable_cnt = positionally_stable(
			self.height_map[x-li+1:x+1, y:y+wi],
			li, wi, hi, self.container_size[2]
		)
		if stable_cnt > 0:
			return True
		return False
	
	def get_valid_actions(self, item_list):
		valid_actions = []
		for item in item_list:
			placement, item_state, height_map, mask = self.get_next_state(item, return_type='numpy')
			if placement is not None:
				for t in range(clip_num):
					position = placement[t, :2].astype(np.int32).tolist()
					if self.is_valid(item, position + [0]):
						valid_actions.append({"item": item, "position": position, "rotation": 0})
					if self.is_valid(item, position + [1]):
						valid_actions.append({"item": item, "position": position, "rotation": 1})
		return valid_actions

cs = ContainerState((10, 10, 10))
cs.get_valid_actions([np.array([2, 3, 4]), np.array([9, 3, 2])])

[{'item': array([2, 3, 4]), 'position': [1, 0], 'rotation': 0},
 {'item': array([2, 3, 4]), 'position': [9, 0], 'rotation': 0},
 {'item': array([2, 3, 4]), 'position': [9, 0], 'rotation': 1},
 {'item': array([2, 3, 4]), 'position': [1, 7], 'rotation': 0},
 {'item': array([2, 3, 4]), 'position': [9, 7], 'rotation': 0},
 {'item': array([2, 3, 4]), 'position': [9, 7], 'rotation': 1},
 {'item': array([2, 3, 4]), 'position': [2, 0], 'rotation': 0},
 {'item': array([2, 3, 4]), 'position': [2, 0], 'rotation': 1},
 {'item': array([2, 3, 4]), 'position': [9, 0], 'rotation': 0},
 {'item': array([2, 3, 4]), 'position': [9, 0], 'rotation': 1},
 {'item': array([2, 3, 4]), 'position': [2, 8], 'rotation': 1},
 {'item': array([2, 3, 4]), 'position': [9, 8], 'rotation': 1},
 {'item': array([9, 3, 2]), 'position': [8, 0], 'rotation': 0},
 {'item': array([9, 3, 2]), 'position': [8, 0], 'rotation': 1},
 {'item': array([9, 3, 2]), 'position': [9, 0], 'rotation': 0},
 {'item': array([9, 3, 2]), 'position': 

In [24]:
class MCTS_Node:
	def __init__(self, state, parent=None):
		self.state = state
		self.parent = parent
		self.children = {}
		self.visit_count = 0
		self.total_reward = 0.0
		self.prior_prob = 0.0  # Prior probability from RL model

	def is_fully_expanded(self, item_list):
		return len(self.children) == len(self.state.get_valid_actions(item_list))

In [ ]:
class MultiContainerState:
	def __init__(self, container_size, item_list):
		n_containers = container_size.shape[0]
		self.container_size = container_size
		self.item_list = copy.deepcopy(item_list)
		self.states = [ContainerState(container_size[i, :]) for i in range(n_containers)]

In [ ]:
def get_outer_reward(multi_env):
	used_volume = 0
	total_volume = 0
	usage_penalty = 0
	for env in multi_env.states:
		used_volume += env.packed_volume
		total_volume += env.total_volume
		if env.packed_volume > 1:
			usage_penalty += 1
	return (used_volume / total_volume) - usage_penalty * 0.1

def get_inner_reward(env):
	return env.packed_volume / env.total_volume

# action = {"item": item, "position": position, "orientation": orientation}

def apply_inner_action(env, action):
	new_env = copy.deepcopy(env)
	new_env.add_item(action["item"], action["position"], action["orientation"])
	return new_env

$ UCB = Q + c \cdot P \cdot \sqrt{\frac{\log(N_{parent})}{1 + N_{child}}} $

In [ ]:
def select_best_child(node, exploration_constant):
	best_score = float('-inf')
	best_child = None

	for action, child in node.children.items():
		ucb = (child.total_reward / (child.visit_count + 1e-8)) + \
			exploration_constant * child.prior_prob * \
			np.sqrt(np.log(node.visit_count + 1) / (child.visit_count + 1e-8))
		if ucb > best_score:
			best_score = ucb
			best_child = child

	return best_child

def expand_node(node, action):
	new_state = apply_inner_action(node.state, action)
	child_node = MCTS_Node(new_state, parent=node)
	node.children[action] = child_node
	return child_node

def backpropagate(node, reward):
	while node is not None:
		node.visit_count += 1
		node.total_reward += reward
		node = node.parent

In [ ]:
def simulate_rollout(env, item_list_, model):
	current_env = env.clone()
	item_list = copy.deepcopy(item_list_)
	used = [False for i in range(len(item_list))]
	used_cnt = 0
	invalid_cnt = 0
	total_reward = 0

	# while not current_state.is_full() and len(current_state.remaining_bins) > 0:
	while used_cnt + invalid_cnt < len(item_list):
		total_reward = 0
		invalid_cnt = 0
		for i, item in enumerate(item_list):
			if used[i] == True:
				continue
			# Predict action probabilities using the policy model
			states = current_env.get_next_state(item)
			action_probs = model.predict(states).detach().numpy()
			# Sample an action
			action_idx = np.random.choice(len(action_probs), p=action_probs)
			action_pos = get_action_from_idx(states[0], action_idx, clip_num)
			action = {"item": item, "position": action_pos[:2], "orientation": action_pos[2]}

			# Validate the action
			if not current_env.is_valid(item, action_pos):
				# break  # Stop the simulation if the action is invalid
				invalid_cnt += 1
				continue
			used[i] = True
			used_cnt += 1
			# Apply the action
			current_state = apply_inner_action(current_state, action)

			# Compute reward
			reward = get_inner_reward(current_state)
			total_reward += reward

	return total_reward

In [ ]:
def inner_MCTS(root, model, n_simulations, item_list):
	for t in range(n_simulations):
		node = root
		while node.is_fully_expanded(item_list) and not node.state.is_full():
			node = select_best_child(node, exploration_constant=1.0)
		if not node.state.is_full():
			for action in node.state.get_valid_actions(item_list):
				if action not in node.children:
					new_state = apply_inner_action(node.state, action, item_list)
					child_node = MCTS_Node(new_state, parent=node)
					node.children[action] = child_node
					break

			# Simulation
		reward = simulate_rollout(node.state, model)

		# Backpropagation
		backpropagate(node, reward)

	# Return the best packing strategy for the container
	best_action = max(root.children.items(), key=lambda item: item[1].visit_count)[0]
	return best_action

In [ ]:
def apply_outer_action(state, container_id, models):
	new_state = copy.deepcopy(state)

	# Get the selected container
	selected_container = new_state.container_size[container_id]

	# Run Inner MCTS to pack bins into the selected container
	inner_root = MCTS_Node(selected_container)
	model = models[container_id]  # Get the model specific to this container size
	best_packing_strategy = inner_MCTS(inner_root, model, num_simulations=100)

	# Apply the best packing strategy to the container
	for action in best_packing_strategy:
		selected_container = apply_inner_action(selected_container, action)

	# Remove packed bins from the remaining bins
	packed_bins = [bin for bin in selected_container.packed_bins]
	new_state.remaining_bins = [bin for bin in new_state.remaining_bins if bin not in packed_bins]

	return new_state

In [ ]:
def outer_MCTS(state, models, num_simulations, exploration_constant):
	root = MCTS_Node(state)
	for _ in range(num_simulations):
		# Selection
		node = root
		while node.is_fully_expanded() and not node.state.is_terminal():
			node = select_best_child(node, exploration_constant)

		# Expansion
		if not node.state.is_terminal():
			valid_actions = node.state.get_valid_actions()  # Container selection
			for action in valid_actions:
				if action not in node.children:
					new_state = apply_outer_action(node.state, action)
					child_node = MCTS_Node(new_state, parent=node)
					node.children[action] = child_node
					break

		# Inner MCTS for the selected container
		selected_container_id = action
		selected_container = node.state.containers[selected_container_id]
		inner_root = MCTS_Node(selected_container)
		inner_reward = inner_MCTS(inner_root, models[selected_container_id], num_simulations=100)

		# Update the global state after packing
		node.state.containers[selected_container_id] = inner_root.state
		reward = get_outer_reward(node.state)

		# Backpropagation
		backpropagate(node, reward)

	# Return the best container selection strategy
	best_action = max(root.children.items(), key=lambda item: item[1].visit_count)[0]
	return best_action